# GPS Data Onboarding 2025-2026

## Summary

Hello! This mini project will take you through some basic ways to work with OpenStreetMaps GPS data and use it to estimate the curvature along the track. After completing this notebook you will be able to:
+ Load data from a `.gpx` file into a `GeoSeries`
+ Understand how and why to project longitude/latitude data with the `geopandas` library
+ Work with the underlying data geometries using the `shapely` library
+ How to compute the curvature numerically

## Background

In the summer of 2025, our team won first place at the NCM Motorsports Park Track. For this mini project, we will take you through a simplified version of some of the work we had to do with the track's GPS data. The goal of this project is to calculate the curvature along the track which is useful for determining the max speed our solar car can take a turn. This helps us make our strategy programs run faster since it limits the search space for possible speeds we could drive.

<img src="images/track_satellite_view.png" width="750">

This is a picture from Google Maps of NCM Motorsports Park's track in which we raced at for FSGP 2025. As you can see from the picture, the track has many curves and different supported routes. The circuit that we follow is linked [here](https://www.racingcircuits.info/north-america/usa/ncm-motorsports-park.html). The first thing we would want to do is get this path that the track follows in a less human and more computer friendly layout. We'll get this data from [OpenStreetMaps](https://www.openstreetmap.org/#map=16/37.00258/-86.36695), an open source online map database that contains geographic and other metadata information. Follow the link and you should see the track on OpenStreetMaps. Right click somewhere on the track, hit query features, and hover your mouse on the raceway number. You should see something like this, but perhaps with a different part of the route highlighted.

<img src="images/osm_query.png" width="750">

Notice that this does not cover the entire circuit that we want. In fact, part of it may be wrong! Nevertheless, we would first collect all of the "ways" (the id numbers like #326452609) for the parts of the track that cover the circuit. Then, we call the Overpass API which gives us a representation of these ways. We've done the ~~extremely challenging~~ job of fetching this data for you. You can find the files in the `data` folder. They come in the `.gpx` file format which you can think of essentially as a list of points where for each point we are given the latitude and longitude. 

The route can then be interpreted as connecting these dots with straight lines. For this reason, there will be a lot more points at curves so the shape of it is correct. But we don't really need this many points to encode the information about the shape of the curve: that's what curvature does. This way we can compress the information about the track even further.

## Setup and Imports

Our first job will be to take these three separate lists of points and turn them into one which actually reflects the circuit we want. First we are going to import all the necessary libraries and setup the file paths for the data we need. Run this cell as is.

In [ ]:
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely

%matplotlib widget

# Get GPX file paths
DIR = os.path.dirname(os.path.abspath("curvature.ipynb"))

BIG_TRACK_PATH = os.path.join(DIR, "data/big_track.gpx")
INNER_EXTENSION_PATH = os.path.join(DIR, "data/inner_extension.gpx")
MULSANNE_CHICANE_PATH = os.path.join(DIR, "data/mulsanne_chicane.gpx")

## Load Data

We will be using the `geopandas` library to interact with the `.gpx` files and their data. The library extends `pandas.DataFrame`s and `pandas.Series` to support geospatial data which is represented internally by different geometry types using the `shapely` library. Both of these libraries will be very important for us to understand. Let's load the files into `geopandas.GeoSeries`s and see how everything looks. Make sure that the points match the shape of the circuit.

In [ ]:
# Load from file into GeoSeries
big_track = gpd.GeoSeries.from_file(BIG_TRACK_PATH, layer="track_points")
inner_extension = gpd.GeoSeries.from_file(INNER_EXTENSION_PATH, layer="track_points")
mulsanne_chicane = gpd.GeoSeries.from_file(MULSANNE_CHICANE_PATH, layer="track_points")

# Plot different parts of the track from openstreetmaps
fig, ax = plt.subplots()
fig.suptitle("Track Segments")

#TODO Plot the points from each GeoSeries and make sure to add labels for the legend
big_track.plot(ax=ax, label="Big Track", color="blue")
inner_extension.plot(ax=ax, label="Inner Extension", color="orange")
mulsanne_chicane.plot(ax=ax, label="Mulsanne Chicane", color="green")

ax.legend()

## Three Datasets -> One Circuit

Now, the challenge is to combine them into a single track appropriately. We want to start just before the chicane, add the chicane points, then take the "inner extension" points, join back to the main track while cutting out the extra points, all the way to the beginning. If you look really closely, you can see that the chicane's start and end overlaps with the main track and the same is true for the "inner extension." We want to find exactly the points that they intersect at and use that to construct a new `GeoSeries` consisting of just the points we want. Write some code to find and print out the intersecting indices.

You can do this by iterating through and looking for matches, but we recommend using `shapely.intersection` to find the shared points since this may come in handy in the future. As a hint, convert the `GeoSeries` to a `shapely.LineString` before using `shapely.intersection`. Your code should print out the indices of `big_track` which intersect with `mulsanne_chicane` and the indices that intersect with `inner_extension`.

In [ ]:
#TODO Find the indices in the big_track dataframe that coincide with mulsane_chicane and inner_extension
from shapely.geometry import LineString

#Convert each GeoSeries into LineString
convert_big_track = LineString(big_track.to_list())
convert_inner_extension = LineString(inner_extension.to_list())
convert_mulsanne_chicane = LineString(mulsanne_chicane.to_list())

#Find where the two routes intersect with the main route
big_intersect_inner = convert_big_track.intersection(convert_inner_extension)
big_intersect_chicane = convert_big_track.intersection(convert_mulsanne_chicane)

#Find the corresponding indices in big_track
chicane_indices = [i for i, p in enumerate(big_track) if any(p.equals(pt) for pt in big_intersect_chicane.geoms)]
inner_indices = [i for i, p in enumerate(big_track) if any(p.equals(pt) for pt in big_intersect_inner.geoms)]

print("Chicane indices:", chicane_indices)
print("Inner extension indices:", inner_indices)


Using these indices, we can combine the parts of the track we want. You should fill out the missing indices with ones based on the intersecting indices from the last bit of code you wrote. We are then going to circular shift the elements of the track by `102` so that the first point lines up with the starting/finish line of the actual track. To verify you did this correctly, you should have `346` points in your track and the first point should have the coordinates `(-86.36857, 37.0010411)`.

In [ ]:
#TODO Replace the `_`s with the correct indices you found
track = pd.concat([
    big_track[:chicane_indices[0]],
    mulsanne_chicane,
    big_track[chicane_indices[1] + 1:inner_indices[0]],
    inner_extension,
    big_track[inner_indices[1] + 1:],
])

#TODO Circular shift the elements back by 102 
track = pd.concat([track.iloc[102:], track.iloc[:102]]).reset_index(drop=True)

# Plot the track
fig, ax = plt.subplots()
fig.suptitle("One Track")
track.plot(ax=ax)

print(len(track), track[0])

## Critical Points

We don't actually want to compute curvature at all the points, only some of the more important ones. We refer to these as critical points. Normally one might programatically determine them or manually go through the track to determine them. To save time and keep everyone on the same page (points), we've listed them out for you. You can think of these points as the minimum number of points we would need to roughly reproduce the track. However, to know the path on curves, we would have to find a way to encode how curvy it is. This is exactly why we need to compute the curvature. You don't need to do anything for this cell

In [ ]:
crit_pts = [0, 1, 20, 22, 23, 27, 28, 36, 37, 46, 50, 58, 71, 88, 89, 104, 108, 115, 116, 127, 132, 141, 153, 166, 172, 180, 181, 188, 190, 197, 206, 209, 231, 244, 246, 279, 281, 315, 322, 327, 332, 335, 341]


fig, ax = plt.subplots()
fig.suptitle("Critical Points")
track.plot(ax=ax)
ax.scatter(*np.array([p.coords for p in track[crit_pts]]).T)

## Computing Curvature

If you have taken a Multivariable Calculus course, in the section Vector Calculus, you probably had to learn how to compute the curvature of a smooth curve. Formally, curvature is defined as $$ \kappa = \left\|\frac{d \vec{T}}{ds} \right\| $$
where $\vec{T}$ is the unit tangent vector and we define curvature as the derivative with respect to $s$, the arc length of the curve. A more intuitive definition is that if you placed a circle tangent to a point and picked a radius that conforms to the curve, the curvature at that point is the reciprocal of the radius of that circle.

<img src="images/curvature.jpeg" width="750">

There are a lot of problems we need to solve to go from our representation of points to something that we can compute the curvature of. Estimating derivatives when you have sample points can always be tricky business, especially when you may need multiple orders of derivatives. To solve this, we will use the points we have around our critical point to locally approximate it with a function.

In particular, we will fit the points to a smoothing B-spline which is nothing more than a polynomial. This way, we can easily take derivatives of it. To get a spline approximation, we need points to sample from around our critical points which may be tricky since some don't have any nearby. To solve this, we will first normalize the points that we have so they are roughly evenly spaced. This way we will be able to get a more balanced approximation.

## Discretization

The `shapely` library offers a function just for this purpose called [`segmentize`](https://shapely.readthedocs.io/en/2.1.1/reference/shapely.segmentize.html) and we recommend that you read its documentation. Let's "segmentize" our track so that points are at most **2.5 meters** apart. But wait! The data we have is in latitude and longitude coordinates. We first need to convert this to "meters". But the problem is not so simple because the Earth isn't flat. Since the track covers a small area, we can locally project the track with minimal error in relative distance calculations.

Luckily, people have already thought about and solved this problem. We can convert between already existing coordinate systems with high accuracy for distance calculations. We will use the [Wisconsin Transverse Mercator](https://epsg.io/3071) projection which is very accurate for mainland USA. `geopandas` already has functionality to do this with the `to_crs` function. Make sure to `set_crs` first to `EPSG:4326` so it knows the given data is in latitude longitude format.

In [ ]:
from shapely import Point

#TODO Project the track to EPSG:3071
projected = track.set_crs("EPSG:4326").to_crs("EPSG:3071")


# Discretize the projected points so they are a max of 2.5m apart
discretized = np.array([Point(p) for p in shapely.LineString(np.hstack([projected, projected.iloc[0]])).segmentize(2.5).coords])[:-1]

## Spline Fitting

Now, our goal is to come up with a routine to take a point and then fit a spline to its local neighborhood. We define the local neighboorhood to be around 100m, so 50m in front and 50m behind the point. To handle the first and last points, we need to take the last 50m and put it at the beginning and similarly put the first 50m at the end. 

We also need a way to parameterize the spline, and since we really want to compute curvature in between two critical points, it makes sense to parameterize the spline in terms of distance. Therefore, we need to calculate the cumulative distance along all these points and use this as our reference for future calculations. We also want the neighborhood to not exceed the region between the two critical points.

Our goal here is to define a function that given some distance along the track, returns a spline fit locally in that area.

In [ ]:
from scipy.interpolate import make_splprep

PADDING = 50.0

# Pads the track 50m in the front and back for the worst case
disc_dist = np.hstack([[0], np.cumulative_sum(shapely.distance(discretized[:-1], discretized[1:]))])
pad_back_ind = np.searchsorted(disc_dist, PADDING) + 1
pad_front_ind = np.searchsorted(disc_dist, disc_dist[-1] - PADDING) - 1
track_padded = np.hstack([discretized[pad_front_ind:], discretized, discretized[:pad_back_ind]])

#TODO Compute the cumulative distance, making sure it starts with zero
cumul_dist = np.hstack([[0], np.cumsum(shapely.distance(track_padded[:-1], track_padded[1:]))])

def spline_approx(t, loc):
    # Gets the spline centered at t, but fitted in points in [t - loc, t + loc]
    start, end = np.searchsorted(cumul_dist, (t - loc, t + loc))
    x = cumul_dist[start:end+1]
    y = np.array([p.coords[0] for p in track_padded[start:end+1]]).T
    spline, _ = make_splprep(y, u=x, s=1.0, k=3)
    return spline

# Tests spline approximation
test = 4400.0

spl = spline_approx(test, 10.0)
pts = spl(np.linspace(test - 1.5 * PADDING, test + 1.5 * PADDING, 150))

fig, ax = plt.subplots()
fig.suptitle("Spline Approximation")
track_proj = track.set_crs(4326).to_crs(3071)
track_proj.plot(ax=ax, label="Original")
ax.scatter(*pts, alpha=0.2, label="Spline")
ax.legend()

## Curvature Formulas

Warning! Lots of math ahead. If you are curious, follow along with the derivation, but if it gets too much, the final equations that we care about for computing curvature are:
$$ \frac{\left|x'(t) y''(t) - x''(t) y'(t)\right|}{({x'(t)}^{2} + {y'(t)}^{2})^{\frac{3}{2}}} = \frac{\left\| \vec{r'}(t) \times \vec{r''}(t) \right\|}{({x'(t)}^{2} + {y'(t)}^{2})^{\frac{3}{2}}} $$

Now, our goal is to compute the curvature between this critical point and the next one. Therefore, we need to find the closest point in between the two distance wise and do our computation based on samples around that point. For the actual curvature computation, since we are in two dimensions, the spline is of the form $$\vec{r}(t) = \left(x(t), y(t)\right) $$

We start with $$ \vec{T}(t) = \frac{1}{\sqrt{{x'(t)}^{2} + {y'(t)}^{2}}}\left(x'(t), y'(t)\right) $$

Using the product rule and chain rule gives us that 
$$ \begin{align*} 
    \frac{d \vec{T}}{dt} &= -\frac{x'(t) x''(t) + y'(t) y''{t}}{\left({x'(t)}^{2} + {y'(t)}^{2}\right)^{\frac{3}{2}}}\left(x'(t), y'(t)\right) + \frac{1}{\sqrt{{x'(t)}^{2} + {y'(t)}^{2}}}\left(x''(t), y''(t)\right) \\
    &= \frac{1}{({x'(t)}^{2} + {y'(t)}^{2})^{\frac{3}{2}}}\left(-x'(t) y'(t) y''(t) + {y'(t)}^{2} x''(t), -x'(t) x''(t) y'(t) + {x'(t)}^{2} y''(t) \right) \\
    &= \frac{x'(t) y''(t) - x''(t) y'(t)}{({x'(t)}^{2} + {y'(t)}^{2})^{\frac{3}{2}}}\left(-y'(t), x'(t)\right)
\end{align*} $$

Now we use the fact that
$$ \kappa = \left\|\frac{d \vec{T}}{ds} \right\| = \left\|\frac{d \vec{T}}{dt} \cdot \frac{dt}{ds} \right\| = \frac{\left\|\frac{d \vec{T}}{dt} \right\|}{\left\| \vec{r'}(t) \right\|}  $$

which gives us that
$$ \frac{\left\|\frac{d \vec{T}}{dt} \right\|}{\left\| \vec{r'}(t) \right\|} = \frac{\left\| \frac{x'(t) y''(t) - x''(t) y'(t)}{({x'(t)}^{2} + {y'(t)}^{2})^{\frac{3}{2}}}\left(-y'(t), x'(t)\right) \right\|}{\left\| \sqrt{{x'(t)}^{2} + {y'(t)}^{2}} \right\|} = \frac{\left|x'(t) y''(t) - x''(t) y'(t)\right|}{({x'(t)}^{2} + {y'(t)}^{2})^{\frac{3}{2}}} = \frac{\left\| \vec{r'}(t) \times \vec{r''}(t) \right\|}{({x'(t)}^{2} + {y'(t)}^{2})^{\frac{3}{2}}} $$

We can use the spline representation to get the first and second order derivatives ($\vec{r'}(t)$ and $\vec{r''}(t)$) and then evaluate them at our desired midpoint to substitute in to the formula above.

We loop through all the critical points, keeping track of the previous one so we have the pair that we would want to compute the curvature of in between. We also need to take on the first critical point again at the end since we would want the curvature in between the end and the first point because then after that would be the next lap.

Each iteration we compute the curvature using the aforementioned formulas, and to visualize it, we consider the following. The first derivative is tangent to the curve, so if we rotate it counterclockwise 90 degrees, then we have a vector normal to the curve. If we turn this into a unit vector, and consider the signed variant of curvature to tell us whether the curve is twisting counterclockwise or clockwise to adjust the direction of the unit normal vector correctly. Then, multiply this by the radius of curvature and we have the center of the circle which represents the curvature at the point.

These circles should fit the curve appropriately and be tangent to the point of which we are computing them.

In [ ]:
from matplotlib.patches import Circle

def cross2d(x, y):
    return x[..., 0] * y[..., 1] - x[..., 1] * y[..., 0]
# Keep track of last critical point; find the index of the first one
curvature = np.zeros(len(crit_pts))
crit_pts_pad = np.hstack([crit_pts, [crit_pts[0]]]) # For the last one, loop around for the first crit point to be the next one
ind_last = 0
while track_padded[ind_last] != track_proj[crit_pts[0]]:
    ind_last += 1

fig, ax = plt.subplots()
fig.suptitle("Curvature Visualized")
track_proj.plot(ax=ax)

for i, p in enumerate(crit_pts_pad[1:]):
    ind_next = ind_last + 1
    # Find next critical point
    while track_padded[ind_next] != track_proj[p]:
        ind_next += 1
    # Distance in between where we want to center our spline approximation
    mid_dist = (cumul_dist[ind_next] + cumul_dist[ind_last]) / 2
    loc = min(mid_dist - cumul_dist[ind_last], PADDING)
    spl = spline_approx(mid_dist, loc)

    # Compute derivatives
    spl_der = spl.derivative()
    spl_2der = spl_der.derivative()

    d1 = spl_der(mid_dist)
    d2 = spl_2der(mid_dist)

    #TODO Compute curvature
    kappa = np.abs(cross2d(d1, d2)) / (np.linalg.norm(d1)**3)

    # Compute normal vector
    drx, dry = d1 / np.linalg.vector_norm(d1, ord=2)
    Nv = np.sign(cross2d(d1, d2)) * np.array([-dry, drx])

    midpt = spl(mid_dist)
    # A straight line can be thought to have an infinite radius of curvature
    # Thus if the curvature is really small, the radius of curvature is very large and so at that point, the line is straight
    if kappa > 6e-3:
        radius = 1 / kappa
        center = midpt + Nv / kappa
        ax.scatter(*midpt, color="orange")
        ax.add_patch(
            Circle(
                center, radius, color="red", fill=True, alpha=0.5, linestyle="dashed"
            )
        )
    else:
        ax.scatter(*midpt, color="green")

    curvature[i] = kappa
    ind_last = ind_next